In [ ]:
### 1. Download e Preparação do Dataset
import polars as pl
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from datasets import Dataset
import os
from dotenv import load_dotenv
from huggingface_hub import login

In [ ]:
# Carregar as variáveis de ambiente do .env
load_dotenv()

# Obter o token
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    raise ValueError("O token da Hugging Face (HF_TOKEN) não foi encontrado no arquivo .env")


In [ ]:
def load_fine_tuning_dataset():
    """
    Carrega o dataset de fine-tuning a partir de um arquivo JSON onde cada linha contém um objeto JSON separado.
    """
    data = pl.read_ndjson("trn.json")
    return data.select(["title", "content"]).fill_null("")

def load_train_test_datasets():
    """
    Carrega os datasets de treino e teste a partir de arquivos TXT.
    """
    dataset = {}
    for key, filename in zip(["train", "test"],
                              ["filter_labels_train.txt", "filter_labels_test.txt"]):
        data = pl.read_csv(filename, separator="\n", has_header=False, new_columns=["text"]).to_pandas()
        dataset[key] = pl.DataFrame({
            "question": data["text"][0::2].reset_index(drop=True), 
            "answer": data["text"][1::2].reset_index(drop=True)
        })
    return dataset


In [ ]:
dataset_fine_tuning = load_fine_tuning_dataset()
dataset_train_test = load_train_test_datasets()

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

In [ ]:
def persist_data_to_chromadb(dataset, batch_size=5000):
    """
    Persiste os dados no ChromaDB de forma síncrona, evitando problemas com asyncio.
    """
    texts = (dataset["title"] + " " + dataset["content"]).to_list()

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        vectorstore.add_texts(batch)  # Insere o lote no ChromaDB

# Exemplo de chamada
persist_data_to_chromadb(dataset_fine_tuning)

In [ ]:
### 3. Execução do Fine-Tuning
def fine_tune_model(dataset):
    """
    Realiza o fine-tuning do modelo Llama 3 sem quantização.
    """
    model_name = "meta-llama/Llama-3.1-8B-Instruct"  # Mantive um modelo menor para rodar melhor em CPU
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    def tokenize_function(examples):
        return tokenizer(examples["content"], padding="max_length", truncation=True)

    train_dataset = Dataset.from_pandas(dataset.to_pandas())
    tokenized_datasets = train_dataset.map(tokenize_function, batched=True)

    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=3,
        per_device_train_batch_size=2,  # Reduzi o batch size para evitar estouro de RAM
        logging_dir="./logs"
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_datasets)
    trainer.train()

    return model

In [ ]:
def retrieve_context(question):
    """
    Recupera contexto relevante do ChromaDB com base na pergunta do usuário.
    """
    docs = vectorstore.similarity_search(question, k=3)
    return " ".join([doc.page_content for doc in docs])

In [ ]:
# Carregar modelo e tokenizer apenas uma vez (fora da função)
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="cpu")

def generate_response(question):
    """
    Gera uma resposta otimizada com base na pergunta do usuário.
    """
    context = retrieve_context(question)
    prompt = f"Contexto: {context}\nPergunta: {question}\nResposta:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cpu")

    # Otimização para evitar cálculos desnecessários
    outputs = model.generate(**inputs, max_new_tokens=100)  # Limitando tokens gerados

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response




In [ ]:
pergunta = "Qual é o melhor fone de ouvido?"
resposta = generate_response(pergunta)
print(f"Resposta: {resposta}")

: 